**오늘의 학습 목표**
- PyMuPDF4LLM 활용하여 글 추출
- 중요한 이미지와 그렇지 않은 이미지 구분

**학습 날짜**
- 2026.06.02

### PyMuPDF4LLM
- PDF 전용 파싱 라이브러리
- 페이지 별로 정보들을 잘 파싱한다.
- llm한테 넘겨줄 때 잘 이해하도록 파싱하는 라이브러리
- 상세한 설정 가능(많은 매개변수)
- 하지만 마크다운과 마찬가지로 복잡한 표, 그림 등은 잘 못한다

Pymupdf4llm 공식문서 링크입니다.

https://pymupdf.readthedocs.io/en/latest/pymupdf4llm/api.html#pymupdf4llm-api

In [2]:
# %pip install pymupdf4llm

In [3]:
import pymupdf4llm
md_text = pymupdf4llm.to_markdown("data/Portfolio_26.pdf")

In [4]:
# 그냥 출력하면 너무 길어서, Pymupdf4llm이 파싱한 결과를 md 파일로 저장
import pathlib
pathlib.Path("pymupdf4llm_AI.md").write_bytes(md_text.encode())

14732

In [5]:
import pymupdf4llm
#파일의 일부분만 추출하여 마크다운 변환할 수 있습니다.
md_text = pymupdf4llm.to_markdown(
    "data/Portfolio_26.pdf",
    pages=[2], 
    hdr_info=False,
    ignore_code=True
)

print(md_text)

WiDiNetWiDiNet: Photometric Stereo Network for Complex Lighting : Photometric Stereo Network for Complex Lighting

using Relighted Images (ICCV 2023, Submitted)using Relighted Images (ICCV 2023, Submitted)
**Photometric Stereo (1)Photometric Stereo (1)**

- • **연구의연구의필요성필요성**

1.1. 2D 2D 데이터가데이터가가지는가지는한계점한계점극복을극복을위해서위해서 **2D 2D 데이터를데이터를3D 3D 데이터로데이터로변환하는변환하는2D2D--toto--3D 3D** 기술의기술의연구연구필요필요

2.2. 예를예를들어들어, 2D, 2D 이미지에서의이미지에서의 **saturation saturation 문제문제** 는는3D 3D 데이터에서데이터에서해결해결가능하므로가능하므로불량불량검출검출정확도를정확도를향상시킴향상시킴

- • **PhotometricPhotometric Stereo …Stereo … ??**

1.1. 2D 2D 데이터를데이터를3D 3D 데이터로데이터로변환하는변환하는 **2D2D--toto--3D3D** 의의대표대표기술기술중중하나하나

2.2. 단일단일시점시점((고정된고정된카메라카메라각도각도))에서에서방향이방향이다른다른조명을조명을통해서통해서찍은찍은표면의표면의여러여러이미지를이미지를캡처하여캡처하여

**표면의표면의각각지점의지점의법선법선(normal)(normal)을을추정추정** 하는하는tasktask

**<Photometric<Photometric StereoStereo Process>Process>**

   - • **기존기존Photometric Stereo Photometric Stereo 모델의모델의문제점문제점**

1.1. 기존의기존의photometric stereophotometric stereo모델모델은은 **1) 1) 암실암실환경에서

#### VLM에 캡셔닝 작업을 전달하기 전에 정보량이 풍부한 이미지만 남기기

- 이제 EasyOCR을 활용해 이미지 속 텍스트의 수를 기준으로 정보량이 풍부한 이미지와 그렇지 않은 이미지를 구분할 수 있습니다.
- 그 이후는 VLM 추론을 통해 이미지 캡션을 수행하면 됩니다.

In [6]:
# %pip install easyocr

In [7]:
# 예시로 주신 데이터가 제일 괜찮은거같아 그대로 활용..
md_data = pymupdf4llm.to_markdown(
    "data/국가별 공공부문 AI 도입 및 활용 전략.pdf",
    page_chunks=True,
    write_images=True,
    image_path="PyMuPDF4LLM_AI",       # 저장 경로 지정
    image_format="png",          # 저장 포맷
    dpi=200,
    image_size_limit=0.02        # 2% 이하 크기 이미지 제외
)

EasyOCR을 통해 이미지 내 글자를 인식하고, 글자수가 충분한 경우에만 VLM에 넘기도록 처리하는 로직입니다.

In [15]:
# 쓸데없는 이미지 거르기 위해 특정 THRESHOLD 측정
import os
from PIL import Image
import numpy as np
import easyocr

# EasyOCR reader
reader = easyocr.Reader(['ko', 'en'], gpu=True)

# 이미지 폴더 경로
image_folder = "PyMuPDF4LLM_AI"

# 결과 저장용 리스트
rich_images = []   # 정보 풍부
poor_images = []   # 정보 부족

# 폴더 내 이미지 순회
for filename in os.listdir(image_folder):
    if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
        img_path = os.path.join(image_folder, filename)

        try:
            # 이미지 불러오기
            img = Image.open(img_path).convert("RGB")
            img_np = np.array(img)

            # OCR 수행
            results = reader.readtext(img_np)

            texts = [res[1] for res in results]
            total_text = ''.join(texts)

            # 조건 검사 (threshold)
            if len(texts) >= 3 and len(total_text) >= 30:
                rich_images.append((filename, len(texts), len(total_text)))
            else:
                poor_images.append((filename, len(texts), len(total_text)))

        except Exception as e:
            print(f"❌ 오류 발생 - {filename}: {e}")

# 결과 출력
print("\n📘 정보량 많은 이미지:")
for name, blocks, chars in rich_images:
    print(f"✔ {name} - 블럭 {blocks}개, 총 글자수 {chars}")

print("\n📄 정보 부족한 이미지:")
for name, blocks, chars in poor_images:
    print(f"✖ {name} - 블럭 {blocks}개, 총 글자수 {chars}")


Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.



📘 정보량 많은 이미지:
✔ 국가별-공공부문-AI-도입-및-활용-전략.pdf-0-4.png - 블럭 9개, 총 글자수 78
✔ 국가별-공공부문-AI-도입-및-활용-전략.pdf-1-0.png - 블럭 6개, 총 글자수 58
✔ 국가별-공공부문-AI-도입-및-활용-전략.pdf-10-0.png - 블럭 4개, 총 글자수 50
✔ 국가별-공공부문-AI-도입-및-활용-전략.pdf-12-0.png - 블럭 52개, 총 글자수 537
✔ 국가별-공공부문-AI-도입-및-활용-전략.pdf-2-0.png - 블럭 4개, 총 글자수 53
✔ 국가별-공공부문-AI-도입-및-활용-전략.pdf-20-0.png - 블럭 6개, 총 글자수 60
✔ 국가별-공공부문-AI-도입-및-활용-전략.pdf-20-1.png - 블럭 4개, 총 글자수 55
✔ 국가별-공공부문-AI-도입-및-활용-전략.pdf-22-0.png - 블럭 4개, 총 글자수 50
✔ 국가별-공공부문-AI-도입-및-활용-전략.pdf-28-5.png - 블럭 4개, 총 글자수 41
✔ 국가별-공공부문-AI-도입-및-활용-전략.pdf-28-6.png - 블럭 4개, 총 글자수 50
✔ 국가별-공공부문-AI-도입-및-활용-전략.pdf-28-7.png - 블럭 5개, 총 글자수 74
✔ 국가별-공공부문-AI-도입-및-활용-전략.pdf-30-0.png - 블럭 4개, 총 글자수 50
✔ 국가별-공공부문-AI-도입-및-활용-전략.pdf-31-0.png - 블럭 41개, 총 글자수 799
✔ 국가별-공공부문-AI-도입-및-활용-전략.pdf-32-0.png - 블럭 57개, 총 글자수 627
✔ 국가별-공공부문-AI-도입-및-활용-전략.pdf-39-0.png - 블럭 6개, 총 글자수 86
✔ 국가별-공공부문-AI-도입-및-활용-전략.pdf-4-0.png - 블럭 5개, 총 글자수 73
✔ 국가별-공공부문-AI-도입-및-활용-전략.pdf-47-1.png - 블럭 12개, 총 글자수 196
✔ 국가별-공공부문-AI-도입-및-활용-전략.pdf-

In [16]:
import os

print("폴더 경로:")
print(os.path.abspath(image_folder))

print("\n파일 개수:")
print(len(os.listdir(image_folder)))

print("\n파일 목록:")
for f in os.listdir(image_folder):
    print(f)

폴더 경로:
c:\Users\junha\Documents\FC_RAG_2025\RAG_Self_Study\CH03.Advanced RAG\01.Indexing\PyMuPDF4LLM_AI

파일 개수:
30

파일 목록:
국가별-공공부문-AI-도입-및-활용-전략.pdf-0-0.png
국가별-공공부문-AI-도입-및-활용-전략.pdf-0-3.png
국가별-공공부문-AI-도입-및-활용-전략.pdf-0-4.png
국가별-공공부문-AI-도입-및-활용-전략.pdf-0-6.png
국가별-공공부문-AI-도입-및-활용-전략.pdf-1-0.png
국가별-공공부문-AI-도입-및-활용-전략.pdf-10-0.png
국가별-공공부문-AI-도입-및-활용-전략.pdf-12-0.png
국가별-공공부문-AI-도입-및-활용-전략.pdf-2-0.png
국가별-공공부문-AI-도입-및-활용-전략.pdf-20-0.png
국가별-공공부문-AI-도입-및-활용-전략.pdf-20-1.png
국가별-공공부문-AI-도입-및-활용-전략.pdf-21-0.png
국가별-공공부문-AI-도입-및-활용-전략.pdf-22-0.png
국가별-공공부문-AI-도입-및-활용-전략.pdf-28-2.png
국가별-공공부문-AI-도입-및-활용-전략.pdf-28-3.png
국가별-공공부문-AI-도입-및-활용-전략.pdf-28-5.png
국가별-공공부문-AI-도입-및-활용-전략.pdf-28-6.png
국가별-공공부문-AI-도입-및-활용-전략.pdf-28-7.png
국가별-공공부문-AI-도입-및-활용-전략.pdf-29-0.png
국가별-공공부문-AI-도입-및-활용-전략.pdf-30-0.png
국가별-공공부문-AI-도입-및-활용-전략.pdf-31-0.png
국가별-공공부문-AI-도입-및-활용-전략.pdf-32-0.png
국가별-공공부문-AI-도입-및-활용-전략.pdf-38-0.png
국가별-공공부문-AI-도입-및-활용-전략.pdf-39-0.png
국가별-공공부문-AI-도입-및-활용-전략.pdf-4-0.png
국가별-공공부문-AI-도입-및-활용-